In [1]:
from pathlib import Path
import pandas as pd
import hashlib
from collections import defaultdict
from tqdm.notebook import tqdm

DATASET_PATH = Path(r"D:\FINAL DFDT\SELECTED_DATASET")
MANIFEST_PATH = DATASET_PATH / "selection_manifest.csv"

manifest = pd.read_csv(MANIFEST_PATH)

print("Manifest records:", len(manifest))
print("Dataset:", DATASET_PATH)

Manifest records: 10000
Dataset: D:\FINAL DFDT\SELECTED_DATASET


In [2]:
missing_files = []

for _, row in manifest.iterrows():
    class_name = str(row["class"]).lower()
    filename = Path(str(row["destination_image"])).name

    current_path = DATASET_PATH / class_name / filename

    if not current_path.exists():
        missing_files.append(str(current_path))

print("Total manifest records:", len(manifest))
print("Missing mapped images:", len(missing_files))

if missing_files:
    print("\nFirst missing files:")
    print(missing_files[:10])
else:
    print("All manifest records correctly map to current images.")

Total manifest records: 10000
Missing mapped images: 0
All manifest records correctly map to current images.


In [5]:
import hashlib
import pandas as pd
from pathlib import Path

hash_records = []

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for class_name in ["real", "fake"]:

    image_files = [
        p for p in (DATASET_PATH / class_name).rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    print(f"\nProcessing {class_name.upper()}: {len(image_files)} images")

    for i, image_path in enumerate(image_files, start=1):

        sha256 = hashlib.sha256()

        with open(image_path, "rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                sha256.update(chunk)

        hash_records.append({
            "class": class_name.upper(),
            "filename": image_path.name,
            "path": str(image_path),
            "sha256": sha256.hexdigest()
        })

        if i % 500 == 0:
            print(f"Processed {i}/{len(image_files)}")

    print(f"Completed {class_name.upper()}")

hash_df = pd.DataFrame(hash_records)

print("\nHASHING COMPLETED")
print("Total hashed images:", len(hash_df))

display(hash_df.head())


Processing REAL: 5000 images
Processed 500/5000
Processed 1000/5000
Processed 1500/5000
Processed 2000/5000
Processed 2500/5000
Processed 3000/5000
Processed 3500/5000
Processed 4000/5000
Processed 4500/5000
Processed 5000/5000
Completed REAL

Processing FAKE: 5000 images
Processed 500/5000
Processed 1000/5000
Processed 1500/5000
Processed 2000/5000
Processed 2500/5000
Processed 3000/5000


Exception ignored in: <function tqdm.__del__ at 0x00000272123EB740>
Traceback (most recent call last):
  File "c:\Users\STEPHEN DHANUSH F\miniconda3\envs\amsf_gnet\Lib\site-packages\tqdm\std.py", line 1154, in __del__
    self.close()
  File "c:\Users\STEPHEN DHANUSH F\miniconda3\envs\amsf_gnet\Lib\site-packages\tqdm\notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
    ^^^^^^^^^
AttributeError: 'tqdm_notebook' object has no attribute 'disp'


Processed 3500/5000
Processed 4000/5000
Processed 4500/5000
Processed 5000/5000
Completed FAKE

HASHING COMPLETED
Total hashed images: 10000


,class,filename,path,sha256
0,REAL,000_01_191.png,D:\FINAL DFDT\SELECTED_DATASET\real\000_01_191...,e9b85389585de8495002c3a0e0e2486a19355c2bc1c615...
1,REAL,000_02_178.png,D:\FINAL DFDT\SELECTED_DATASET\real\000_02_178...,9f2a3b834d1b464bae1b92d84b89c874753c26267a8e67...
2,REAL,000_03_101.png,D:\FINAL DFDT\SELECTED_DATASET\real\000_03_101...,b17910aa40a5f85d80c5f93be9aee4d61ef0d1f968c8ee...
3,REAL,000_04_076.png,D:\FINAL DFDT\SELECTED_DATASET\real\000_04_076...,d1f1f48dc08adf670976b6ae5dc6f4dcb37474deba9a1d...
4,REAL,000_05_063.png,D:\FINAL DFDT\SELECTED_DATASET\real\000_05_063...,41b41efefe2af456cef9973a432e54453663206650478e...


In [6]:
results = []

for class_name in ["real", "fake"]:
    class_path = DATASET_PATH / class_name

    if not class_path.exists():
        continue

    image_files = [
        p for p in class_path.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    corrupted = []

    for image_path in image_files:
        try:
            with Image.open(image_path) as img:
                img.verify()
        except Exception:
            corrupted.append(str(image_path))

    results.append({
        "Class": class_name.upper(),
        "Total Images": len(image_files),
        "Corrupted Images": len(corrupted),
        "Readable Images": len(image_files) - len(corrupted)
    })

    print(f"\n{class_name.upper()} corrupted files:")
    print(corrupted[:10])

inspection_df = pd.DataFrame(results)
display(inspection_df)


REAL corrupted files:
['D:\\FINAL DFDT\\SELECTED_DATASET\\real\\000_01_191.png', 'D:\\FINAL DFDT\\SELECTED_DATASET\\real\\000_02_178.png', 'D:\\FINAL DFDT\\SELECTED_DATASET\\real\\000_03_101.png', 'D:\\FINAL DFDT\\SELECTED_DATASET\\real\\000_04_076.png', 'D:\\FINAL DFDT\\SELECTED_DATASET\\real\\000_05_063.png', 'D:\\FINAL DFDT\\SELECTED_DATASET\\real\\001_01_399.png', 'D:\\FINAL DFDT\\SELECTED_DATASET\\real\\001_02_029.png', 'D:\\FINAL DFDT\\SELECTED_DATASET\\real\\001_03_014.png', 'D:\\FINAL DFDT\\SELECTED_DATASET\\real\\001_04_074.png', 'D:\\FINAL DFDT\\SELECTED_DATASET\\real\\001_05_192.png']

FAKE corrupted files:
['D:\\FINAL DFDT\\SELECTED_DATASET\\fake\\000_003_01_356.png', 'D:\\FINAL DFDT\\SELECTED_DATASET\\fake\\000_003_02_293.png', 'D:\\FINAL DFDT\\SELECTED_DATASET\\fake\\000_003_03_140.png', 'D:\\FINAL DFDT\\SELECTED_DATASET\\fake\\000_003_04_203.png', 'D:\\FINAL DFDT\\SELECTED_DATASET\\fake\\000_003_05_191.png', 'D:\\FINAL DFDT\\SELECTED_DATASET\\fake\\001_870_01_459.png', 

,Class,Total Images,Corrupted Images,Readable Images
0,REAL,5000,5000,0
1,FAKE,5000,5000,0


In [7]:
manifest_path = DATASET_PATH / "selection_manifest.csv"

if manifest_path.exists():
    manifest = pd.read_csv(manifest_path)

    print("Manifest found!")
    print("Total records:", len(manifest))
    print("\nColumns:")
    print(manifest.columns.tolist())

    display(manifest.head())
else:
    print("selection_manifest.csv not found.")

Manifest found!
Total records: 10000

Columns:
['class', 'source_subfolder', 'source_image', 'destination_image']


,class,source_subfolder,source_image,destination_image
0,REAL,000,E:\FINAL DATA1\real\000\191.png,E:\FINAL DATA1\SELECTED_DATASET\real\000_01_19...
1,REAL,000,E:\FINAL DATA1\real\000\178.png,E:\FINAL DATA1\SELECTED_DATASET\real\000_02_17...
2,REAL,000,E:\FINAL DATA1\real\000\101.png,E:\FINAL DATA1\SELECTED_DATASET\real\000_03_10...
3,REAL,000,E:\FINAL DATA1\real\000\076.png,E:\FINAL DATA1\SELECTED_DATASET\real\000_04_07...
4,REAL,000,E:\FINAL DATA1\real\000\063.png,E:\FINAL DATA1\SELECTED_DATASET\real\000_05_06...


In [8]:
duplicate_hashes = hash_df[
    hash_df.duplicated(subset="sha256", keep=False)
].sort_values("sha256")

cross_class_duplicates = []

for hash_value, group in duplicate_hashes.groupby("sha256"):
    classes = set(group["class"])

    if len(classes) > 1:
        cross_class_duplicates.append({
            "sha256": hash_value,
            "classes": list(classes),
            "files": group["filename"].tolist()
        })

print("Total duplicate image records:", len(duplicate_hashes))
print("Unique duplicate hashes:", duplicate_hashes["sha256"].nunique())
print("Cross-class duplicate groups:", len(cross_class_duplicates))

display(duplicate_hashes.head(20))

Total duplicate image records: 0
Unique duplicate hashes: 0
Cross-class duplicate groups: 0


,class,filename,path,sha256


In [9]:
source_summary = (
    manifest.groupby("class")
    .agg(
        total_images=("source_image", "count"),
        unique_source_folders=("source_subfolder", "nunique")
    )
    .reset_index()
)

print("SOURCE GROUP SUMMARY")
display(source_summary)

print("\nImages per source folder:")
folder_distribution = (
    manifest.groupby(["class", "source_subfolder"])
    .size()
    .reset_index(name="image_count")
)

display(
    folder_distribution.groupby(["class", "image_count"])
    .size()
    .reset_index(name="number_of_folders")
)

SOURCE GROUP SUMMARY


,class,total_images,unique_source_folders
0,FAKE,5000,999
1,REAL,5000,999



Images per source folder:


,class,image_count,number_of_folders
0,FAKE,2,1
1,FAKE,5,990
2,FAKE,6,8
3,REAL,5,994
4,REAL,6,5


In [10]:
REPORT_PATH = DATASET_PATH / "verification_reports"
REPORT_PATH.mkdir(exist_ok=True)

hash_df.to_csv(REPORT_PATH / "image_hashes.csv", index=False)
duplicate_hashes.to_csv(
    REPORT_PATH / "exact_duplicates.csv", index=False
)

folder_distribution.to_csv(
    REPORT_PATH / "source_folder_distribution.csv", index=False
)

print("Verification reports saved:", REPORT_PATH)

Verification reports saved: D:\FINAL DFDT\SELECTED_DATASET\verification_reports


In [11]:
# Source folder distribution

folder_distribution = (
    manifest.groupby(["class", "source_subfolder"])
    .size()
    .reset_index(name="image_count")
)

print("IMAGE DISTRIBUTION PER SOURCE FOLDER")
display(
    folder_distribution.groupby(["class", "image_count"])
    .size()
    .reset_index(name="number_of_folders")
)

print("\nMINIMUM IMAGES PER FOLDER:")
print(folder_distribution.groupby("class")["image_count"].min())

print("\nMAXIMUM IMAGES PER FOLDER:")
print(folder_distribution.groupby("class")["image_count"].max())

print("\nAVERAGE IMAGES PER FOLDER:")
print(folder_distribution.groupby("class")["image_count"].mean())

IMAGE DISTRIBUTION PER SOURCE FOLDER


,class,image_count,number_of_folders
0,FAKE,2,1
1,FAKE,5,990
2,FAKE,6,8
3,REAL,5,994
4,REAL,6,5



MINIMUM IMAGES PER FOLDER:
class
FAKE    2
REAL    5
Name: image_count, dtype: int64

MAXIMUM IMAGES PER FOLDER:
class
FAKE    6
REAL    6
Name: image_count, dtype: int64

AVERAGE IMAGES PER FOLDER:
class
FAKE    5.005005
REAL    5.005005
Name: image_count, dtype: float64


In [12]:
print("Missing manifest mappings:", len(missing_files))
print("Total hashed images:", len(hash_df))

Missing manifest mappings: 0
Total hashed images: 10000


In [13]:
real_groups = set(
    manifest.loc[
        manifest["class"].str.upper() == "REAL",
        "source_subfolder"
    ].astype(str)
)

fake_groups = set(
    manifest.loc[
        manifest["class"].str.upper() == "FAKE",
        "source_subfolder"
    ].astype(str)
)

common_groups = real_groups.intersection(fake_groups)

print("REAL folders:", len(real_groups))
print("FAKE folders:", len(fake_groups))
print("Common folder IDs:", len(common_groups))
print("REAL-only folders:", len(real_groups - fake_groups))
print("FAKE-only folders:", len(fake_groups - real_groups))
print("Sample common IDs:", sorted(common_groups)[:20])

REAL folders: 999
FAKE folders: 999
Common folder IDs: 0
REAL-only folders: 999
FAKE-only folders: 999
Sample common IDs: []
